SETUP

In [ ]:
#Change the tensorflow import bassed on OS and GPU support
pip install tensorflow[and-cuda] opencv-python matplotlib

: 

In [3]:
pip list

Package                      Version
---------------------------- ------------
absl-py                      2.4.0
aiofiles                     25.1.0
asttokens                    3.0.1
astunparse                   1.6.3
certifi                      2026.2.25
charset-normalizer           3.4.4
comm                         0.2.3
contourpy                    1.3.2
cycler                       0.12.1
debugpy                      1.8.20
decorator                    5.2.1
etils                        1.13.0
exceptiongroup               1.3.1
executing                    2.2.1
flatbuffers                  25.12.19
flax                         0.10.7
fonttools                    4.61.1
fsspec                       2026.2.0
gast                         0.7.0
google-pasta                 0.2.0
grpcio                       1.78.0
h5py                         3.15.1
humanize                     4.15.0
idna                         3.11
importlib_resources          6.5.2
ipykernel                   

In [4]:
import tensorflow as tf
import os 

BATCH_SIZE = 16
SEED =  1

2026-03-05 23:23:34.711917: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-05 23:23:36.005880: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772774616.428905   30222 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772774616.541385   30222 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772774617.521274   30222 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [5]:
gpus = tf.config.experimental.list_physical_devices('GPU')

In [6]:
gpus

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

In [7]:
#avoid oom error
gpus = tf.config.experimental.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

In [8]:
from tensorflow.keras.applications import EfficientNetV2L

model = EfficientNetV2L(weights = 'imagenet', include_top=False, input_shape=(224,224,3))

I0000 00:00:1772774717.096788   30222 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5563 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


In [9]:
#model.summary()

In [10]:
for layer in model.layers[-35:]: #unlock after base trainging is done [-10:] unlocks 10 layers
    layer.trainable = False

In [11]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import *

my_model = Sequential([model,
                        Conv2D(256, 3, 1, 
                        activation = 'relu'),
                        GlobalAveragePooling2D(), 
                        Dense(256, activation = 'relu'),
                        Dropout(0.2),
                        Dense(256, activation = 'relu'),
                        Dropout(0.2), 
                        Dense(525, activation = 'softmax')])
my_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetv2-l (Functional)   │ (None, 7, 7, 1280)     │   117,746,848 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 5, 5, 256)      │     2,949,376 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 256)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 525)            │       134,925 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 120,962,733 (461.44 MB)

 Trainable params: 107,229,997 (409.05 MB)

 Non-trainable params: 13,732,736 (52.39 MB)

In [12]:
data = tf.keras.utils.image_dataset_from_directory('data/BirdData', color_mode = 'rgb', batch_size = BATCH_SIZE, image_size = (224,224), shuffle = True, validation_split = 0.2, subset = 'training', seed = SEED)
Valdata = tf.keras.utils.image_dataset_from_directory('data/ValData', color_mode = 'rgb', batch_size = BATCH_SIZE, image_size = (224,224), shuffle = True, validation_split = 0.2, subset = 'validation', seed = SEED)
Testdata = tf.keras.utils.image_dataset_from_directory('data/TestData', color_mode = 'rgb', batch_size = BATCH_SIZE, image_size = (224,224), shuffle = False, seed = SEED)

Found 84635 files belonging to 525 classes.
Using 67708 files for training.
Found 2625 files belonging to 525 classes.
Using 525 files for validation.
Found 2625 files belonging to 525 classes.


In [13]:
import matplotlib.pyplot as plt
import numpy as np

In [14]:
from tensorflow.image import flip_left_right, adjust_brightness, adjust_contrast

def augment(image, label):
    image = flip_left_right(image)
    image = adjust_brightness(image, delta = 0.1)
    image = adjust_contrast(image, contrast_factor = 1.75)

    return (image, label)

data = data.map(augment, num_parallel_calls=tf.data.AUTOTUNE)

In [15]:
from tensorflow.keras.applications.efficientnet_v2 import preprocess_input

def preprocess(image, label):
    image = preprocess_input(image)
    return image, label

data = data.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
Valdata = Valdata.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)

In [16]:
data = data.prefetch(tf.data.AUTOTUNE)
data

<_PrefetchDataset element_spec=(TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int32, name=None))>

In [17]:
Valdata = Valdata.prefetch(tf.data.AUTOTUNE)
Valdata

<_PrefetchDataset element_spec=(TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int32, name=None))>

In [18]:
#DONT RUN FIRST TIME
from tensorflow.keras.models import load_model
my_model = load_model(os.path.join('models', 'BirdAI-v9-89.keras'))

In [19]:
from tensorflow.keras.metrics import *

from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import SparseCategoricalCrossentropy

metrics = [
    'accuracy',
    AUC(multi_label=True)
]

my_model.compile(
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    optimizer=tf.keras.optimizers.Adam(1e-4), #change learning rate
    metrics=['accuracy']
)

In [20]:
from tensorflow.keras.callbacks import EarlyStopping

es = EarlyStopping(patience=25, monitor='val_loss', restore_best_weights= True)

In [21]:
hist = my_model.fit(data, epochs = 2, validation_data = Valdata, callbacks = [es])

Epoch 1/2


I0000 00:00:1772774829.789918   31615 service.cc:152] XLA service 0x7b0ca0003fa0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1772774829.790086   31615 service.cc:160]   StreamExecutor device (0): NVIDIA GeForce RTX 4060 Laptop GPU, Compute Capability 8.9
2026-03-05 23:27:12.701242: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1772774851.035734   31615 cuda_dnn.cc:529] Loaded cuDNN version 90300
2026-03-05 23:27:48.525716: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_47812', 4 bytes spill stores, 4 bytes spill loads

2026-03-05 23:27:51.416251: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion

4231/4232 ━━━━━━━━━━━━━━━━━━━━ 0s 117ms/step - accuracy: 0.8700 - loss: 0.4563

2026-03-05 23:37:33.367786: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_47812', 8 bytes spill stores, 8 bytes spill loads

2026-03-05 23:37:34.099501: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_48074', 288 bytes spill stores, 288 bytes spill loads

2026-03-05 23:37:35.107874: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_48074', 52 bytes spill stores, 52 bytes spill loads

2026-03-05 23:37:35.218013: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_48074', 444 bytes spill stores, 444 bytes spill loads

2026-03-05 23:37:35.348504: I exte

4232/4232 ━━━━━━━━━━━━━━━━━━━━ 0s 134ms/step - accuracy: 0.8700 - loss: 0.4563

2026-03-05 23:38:54.808464: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_8395', 8 bytes spill stores, 8 bytes spill loads



4232/4232 ━━━━━━━━━━━━━━━━━━━━ 773s 147ms/step - accuracy: 0.8675 - loss: 0.4650 - val_accuracy: 0.8895 - val_loss: 0.4358
Epoch 2/2


2026-03-05 23:39:15.904806: W tensorflow/core/kernels/data/prefetch_autotuner.cc:52] Prefetch autotuner tried to allocate 9634048 bytes after encountering the first element of size 9634048 bytes.This already causes the autotune ram budget to be exceeded. To stay within the ram budget, either increase the ram budget or reduce element size


4232/4232 ━━━━━━━━━━━━━━━━━━━━ 455s 107ms/step - accuracy: 0.8666 - loss: 0.4628 - val_accuracy: 0.8800 - val_loss: 0.4502


In [ ]:
fig = plt.figure()
plt.plot(hist.history['loss'], color = 'teal', label = 'loss')
plt.plot(hist.history['val_loss'], color = 'orange', label = 'val_loss')
fig.suptitle('loss', fontsize=20)
plt.legend(loc = "upper left")
plt.show()

In [ ]:
fig = plt.figure()
plt.plot(hist.history['accuracy'], color = 'teal', label = 'accuracy')
plt.plot(hist.history['val_accuracy'], color = 'orange', label = 'val_accuracy')
fig.suptitle('accuracy', fontsize=20)
plt.legend(loc = "upper left")
plt.show()

In [ ]:
from tensorflow.keras.models import load_model

In [ ]:
#save model
my_model.save(os.path.join('models', 'BirdAI-v10.keras'))

TypeError: join() got an unexpected keyword argument 'include_optimizer'

In [ ]:
#load model
Loadmodel = load_model(os.path.join('models', 'BirdAI-v9-89.keras'))

In [ ]:
import cv2
import imghdr

#open image
img = cv2.imread('Bald Eagle.jpg')
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.show()

In [ ]:
from tensorflow.keras.applications.efficientnet_v2 import preprocess_input
#resize image
resize = tf.image.resize(img, (224,224))
resize = preprocess_input(resize)

In [ ]:
#tests with the model
yhat = Loadmodel.predict(np.expand_dims(resize, 0))
yhat

In [ ]:
max_idx = np.argmax(yhat)
max_idx
confidence_score = yhat[[max_idx]] * 100
confidence_score

In [ ]:
#print name of prediction
data_dir = 'data/BirdData'
birdnames = []
for image_class in os.listdir(data_dir):
    birdnames.append(image_class)
birdnames = sorted(os.listdir(data_dir))
print(birdnames[max_idx])

In [ ]:
#Convert? not working
import tensorflowjs as tfjs
import os

tfjs.converters.convert_tf_saved_model('BirdAI-v10', 'JSModel')